# Concept detection after debiasing

Computes CAV (LR and Diff-Means) per layer, removes the concept direction via
orthogonal projection, then re-trains classifiers to measure how much concept
information remains.

**Change only `CONCEPT`** — everything else follows automatically.

Outputs:
- `notebooks/results/single_debias/{CONCEPT}/debiased_detection/` — per-layer CSVs + plot
- `data/single_debiasing/{CONCEPT}/cavs/layer_XX.npz` — CAV vectors (keys: `lr_cav`, `dm_cav`)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from xgboost import XGBClassifier
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from software.metrics import get_metrics
from software.torch_lr import TorchLR
from software.viz import plot_debiased_detection

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPT        = 'eyeglasses'
NUM_LAYERS     = 24
RUNS_PER_LAYER = 10

# Solver for logistic regression (both CAV training and probing):
#   'torch_lr' — TorchLR (LBFGS on GPU, stable, recommended)
#   'sgd'      — SGDClassifier from sklearn (stochastic gradient descent)
SOLVER   = 'torch_lr'
SOLVER_C = 0.1

METADATA_PATH   = ROOT / 'data' / 'metadata.csv'
ACTIVATIONS_DIR = ROOT / 'data' / 'activations' / 'raw'
CAV_OUT_DIR     = ROOT / 'data' / 'activations' / 'debiased' / 'single' / CONCEPT / 'cavs_analysis'
OUT_DIR = ROOT / 'notebooks' / 'results' / 'single_debias' / CONCEPT / 'debiased_detection'
CAV_OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv(METADATA_PATH)[['filename', 'split', CONCEPT]]
meta_train = meta[meta['split'] == 'train'][['filename', CONCEPT]]
meta_test  = meta[meta['split'] == 'test'][['filename',  CONCEPT]]

print(f'Concept : {CONCEPT}')
print(f'Solver  : {SOLVER} (C={SOLVER_C})')
print(f'Output  : {OUT_DIR}')
print(f'CAVs    : {CAV_OUT_DIR}')

In [ ]:
def make_clf(random_state=None):
    if SOLVER == 'torch_lr':
        return TorchLR(C=SOLVER_C, max_iter=500, random_state=random_state)
    return SGDClassifier(
        loss='log_loss', penalty='l2',
        alpha=1.0 / (2.0 * SOLVER_C),
        max_iter=1000, tol=1e-4,
        random_state=random_state,
    )


def load_layer(split_label, layer_idx):
    path = ACTIVATIONS_DIR / split_label / f'layer_{layer_idx:02d}.parquet'
    assert path.exists(), f'Missing: {path}. Run 02_get_activations.ipynb first.'
    df = pd.read_parquet(path)
    meta_split = meta_train if split_label == 'train' else meta_test
    df = df.merge(meta_split, on='filename')
    feat_cols = [c for c in df.columns if c not in ('filename', CONCEPT)]
    return df[feat_cols].values.astype(np.float32), df[CONCEPT].astype(int).values


def get_debiased_datasets(X_train, y_train, X_test, X_oob=None):
    """Trains LR-CAV and DM-CAV, returns orthogonally projected datasets.

    If X_oob is provided, the returned dict values are 3-tuples
    (X_train_proj, X_test_proj, X_oob_proj); otherwise 2-tuples.
    """
    # LR CAV
    lr_finder = make_clf(random_state=42)
    lr_finder.fit(X_train, y_train)
    cav_lr = lr_finder.coef_[0]
    cav_lr_norm = cav_lr / (np.linalg.norm(cav_lr) + 1e-10)

    # DM CAV
    cav_dm = X_train[y_train == 1].mean(axis=0) - X_train[y_train == 0].mean(axis=0)
    cav_dm_norm = cav_dm / (np.linalg.norm(cav_dm) + 1e-10)

    def project_out(X, v):
        return X - np.dot(X, v).reshape(-1, 1) * v

    if X_oob is None:
        debiased = {
            'lr': (project_out(X_train, cav_lr_norm), project_out(X_test, cav_lr_norm)),
            'dm': (project_out(X_train, cav_dm_norm), project_out(X_test, cav_dm_norm)),
        }
    else:
        debiased = {
            'lr': (project_out(X_train, cav_lr_norm),
                   project_out(X_test,  cav_lr_norm),
                   project_out(X_oob,   cav_lr_norm)),
            'dm': (project_out(X_train, cav_dm_norm),
                   project_out(X_test,  cav_dm_norm),
                   project_out(X_oob,   cav_dm_norm)),
        }
    return debiased, cav_lr_norm.astype(np.float32), cav_dm_norm.astype(np.float32)

In [ ]:
for layer in tqdm(range(NUM_LAYERS), desc='Layers'):
    X_train_raw, y_train_raw = load_layer('train', layer)
    X_test_raw,  y_test_raw  = load_layer('test',  layer)

    # #3 — CAVs are trained on RAW activations to keep them consistent
    # with the rest of the pipeline (notebook 05). The StandardScaler
    # is used ONLY for reporting probe-classifier metrics, never for CAV.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled  = scaler.transform(X_test_raw)

    n_train = len(y_train_raw)
    layer_results = []
    cav_lr_accum = np.zeros(X_train_raw.shape[1], dtype=np.float32)
    cav_dm_accum = np.zeros(X_train_raw.shape[1], dtype=np.float32)
    rng_layer = np.random.RandomState(layer + 1000)

    for run_id in tqdm(range(RUNS_PER_LAYER), desc=f'L{layer:02d}', leave=False):
        # Bootstrap: indices into the SAME ordering for raw and scaled.
        idx_b = np.random.RandomState(run_id).choice(n_train, size=n_train, replace=True)
        oob_mask = np.ones(n_train, dtype=bool); oob_mask[np.unique(idx_b)] = False

        X_b_raw   = X_train_raw[idx_b]
        X_b_sc    = X_train_scaled[idx_b]
        y_b       = y_train_raw[idx_b]
        X_oob_sc  = X_train_scaled[oob_mask]
        y_oob     = y_train_raw[oob_mask]
        use_oob = (len(y_oob) > 0) and (len(np.unique(y_oob)) == 2)

        # CAVs from RAW activations (#3) — both for the projection used by
        # downstream probes (we project the scaled X for the probe metric)
        # AND for the saved CAV in .npz (which is the raw-space CAV).
        debiased_sc, cav_lr_raw, cav_dm_raw = get_debiased_datasets(X_b_raw, y_b, X_test_raw)
        # For probe metrics in scaled space we project the SCALED X using a
        # CAV trained in scaled space too. Pass X_oob_sc to also obtain the
        # held-out (OOB) projection consistent with the bootstrap projection.
        debiased_for_probe, _, _ = get_debiased_datasets(
            X_b_sc, y_b, X_test_scaled, X_oob=X_oob_sc if use_oob else None,
        )

        cav_lr_accum += cav_lr_raw
        cav_dm_accum += cav_dm_raw

        for method, packed in debiased_for_probe.items():
            if use_oob:
                X_tr_clean, X_te_clean, X_oob_clean = packed
            else:
                X_tr_clean, X_te_clean = packed
                X_oob_clean = None
            probe_models = {
                'LogisticRegression': make_clf(random_state=run_id),
                'XGBoost': XGBClassifier(
                    n_estimators=100, max_depth=4,
                    tree_method='hist', device='cuda',
                    random_state=run_id,
                ),
            }
            for model_name, model in probe_models.items():
                model.fit(X_tr_clean, y_b)
                # #7 — train metric on OOB (held-out from the bootstrap) when
                # available; otherwise fall back to in-sample fit metric, but
                # name the columns fit_* to avoid implying held-out accuracy.
                if use_oob:
                    fit_m = get_metrics(
                        y_oob, model.predict(X_oob_clean),
                        model.predict_proba(X_oob_clean)[:, 1],
                    )
                    fit_prefix = 'train'  # OOB is a valid train-side held-out estimate
                else:
                    fit_m = get_metrics(
                        y_b, model.predict(X_tr_clean),
                        model.predict_proba(X_tr_clean)[:, 1],
                    )
                    fit_prefix = 'fit'
                te_m = get_metrics(y_test_raw, model.predict(X_te_clean),
                                   model.predict_proba(X_te_clean)[:, 1])
                layer_results.append({
                    'layer_id': layer, 'run_id': run_id,
                    'debias_method': method, 'model': model_name,
                    'method': 'real',
                    **{f'{fit_prefix}_{k}': v for k, v in fit_m.items()},
                    **{f'test_{k}':  v for k, v in te_m.items()},
                })

        # ── #14 baselines: random_shuffle (LR on permuted labels) and
        # random_unit_vector (random direction projection)
        for method in ('lr', 'dm'):
            packed = debiased_for_probe[method]
            if use_oob:
                X_tr_clean, X_te_clean, X_oob_clean = packed
            else:
                X_tr_clean, X_te_clean = packed
                X_oob_clean = None

            y_shuf = rng_layer.permutation(y_b)
            clf_shuf = make_clf(random_state=run_id)
            clf_shuf.fit(X_tr_clean, y_shuf)
            if use_oob:
                fit_m = get_metrics(
                    y_oob, clf_shuf.predict(X_oob_clean),
                    clf_shuf.predict_proba(X_oob_clean)[:, 1],
                )
                fit_prefix = 'train'
            else:
                fit_m = get_metrics(
                    y_b, clf_shuf.predict(X_tr_clean),
                    clf_shuf.predict_proba(X_tr_clean)[:, 1],
                )
                fit_prefix = 'fit'
            te_m = get_metrics(y_test_raw, clf_shuf.predict(X_te_clean),
                               clf_shuf.predict_proba(X_te_clean)[:, 1])
            layer_results.append({
                'layer_id': layer, 'run_id': run_id,
                'debias_method': method, 'model': 'LogisticRegression',
                'method': 'random_shuffle',
                **{f'{fit_prefix}_{k}': v for k, v in fit_m.items()},
                **{f'test_{k}':  v for k, v in te_m.items()},
            })

            d = X_tr_clean.shape[1]
            w_rand = rng_layer.randn(d).astype(np.float32)
            w_rand /= (np.linalg.norm(w_rand) + 1e-12)
            def _score(X, w=w_rand):
                z = X @ w
                p = 1.0 / (1.0 + np.exp(-z))
                return (z > 0).astype(int), p
            yhat_b, p_b = _score(X_tr_clean)
            yhat_t, p_t = _score(X_te_clean)
            if use_oob:
                yhat_o, p_o = _score(X_oob_clean)
                fit_m = get_metrics(y_oob, yhat_o, p_o)
                fit_prefix = 'train'
            else:
                fit_m = get_metrics(y_b, yhat_b, p_b)
                fit_prefix = 'fit'
            te_m = get_metrics(y_test_raw, yhat_t, p_t)
            layer_results.append({
                'layer_id': layer, 'run_id': run_id,
                'debias_method': method, 'model': 'LogisticRegression',
                'method': 'random_unit_vector',
                **{f'{fit_prefix}_{k}': v for k, v in fit_m.items()},
                **{f'test_{k}':  v for k, v in te_m.items()},
            })

    pd.DataFrame(layer_results).to_csv(OUT_DIR / f'{layer:02d}_debiased_results.csv', index=False)

    # #4 — Average CAVs from runs and renormalize to unit length.
    lr_cav_final = cav_lr_accum / RUNS_PER_LAYER
    lr_cav_final = lr_cav_final / (np.linalg.norm(lr_cav_final) + 1e-12)
    dm_cav_final = cav_dm_accum / RUNS_PER_LAYER
    dm_cav_final = dm_cav_final / (np.linalg.norm(dm_cav_final) + 1e-12)
    assert np.allclose(np.linalg.norm(lr_cav_final), 1.0, atol=1e-5)
    assert np.allclose(np.linalg.norm(dm_cav_final), 1.0, atol=1e-5)

    np.savez(
        CAV_OUT_DIR / f'layer_{layer:02d}.npz',
        lr_cav=lr_cav_final.astype(np.float32),
        dm_cav=dm_cav_final.astype(np.float32),
    )

## Visualization

In [ ]:
import glob

all_files = sorted(glob.glob(str(OUT_DIR / '*_debiased_results.csv')))
df_all = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True).sort_values('layer_id')

# Main figure: real method only.
df_real = df_all[df_all['method'] == 'real'] if 'method' in df_all.columns else df_all

# #5 — split on first underscore so 'train_roc_auc' parses correctly.
metric_cols = [
    c for c in df_real.columns
    if (c.startswith('train_') or c.startswith('test_'))
    and c.split('_', 1)[1] not in ('recall', 'precision')
]
id_vars = [c for c in ['layer_id', 'model', 'run_id', 'debias_method', 'method'] if c in df_real.columns]

df_melted = df_real.melt(id_vars=id_vars, value_vars=metric_cols,
                         var_name='metric_type', value_name='score')
df_melted['subset'] = df_melted['metric_type'].str.split('_', n=1).str[0]
df_melted['metric'] = df_melted['metric_type'].str.split('_', n=1).str[1]

plot_debiased_detection(
    df_melted, CONCEPT,
    save_path=OUT_DIR / f'debiased_detection_{CONCEPT}.png',
)